In [1]:
pip install qiskit==2.3.0 qiskit-aer

Note: you may need to restart the kernel to use updated packages.


In [2]:
from qiskit import *
from qiskit_aer import AerSimulator
import numpy as np

In [3]:
def Bell(qc,q1,q2):
    qc.h(q1) 
    # |q1> = (|0>+|1>)/sqrt(2) & |q2> = |0>
    qc.cx(q1,q2) 
    # |q1,q2> = (|00>+|11>)/sqrt(2)
    return qc

In [4]:
def correction (qc, bc1, bc2, qbit):
    match (bc1,bc2):
        case (0,0):
            return qc
        case (0,1):
            qc.x(2)
        case (1,0):
            qc.z(2)
        case (1,1):
            qc.x(2)
            qc.z(2)
    return qc

#hay una forma mas eficiente de hacerlo pero se queda aqui como ejemplo visual

In [19]:
def teleport():
    c = ClassicalRegister(3)
    qreg = QuantumRegister(3)
    # se crea un registro clásico para poder acceder a los valores individuales de los bits
    qc = QuantumCircuit(qreg,c)
    # se crea un circuito con el registro clasico y 3 qubits, para que esten unidos

    qc = Bell(qc,1,2)
    qc.cx(0,1)
    qc.h(0)
    qc.measure([0,1],[0,1])
    #En función del resultado de c0,c1 hay que aplicar una corrección en q0
    with qc.if_test((c[0], 1)):
        qc.z(2)

    with qc.if_test((c[1], 1)):
        qc.x(2)

    qc.measure(2,2)
    return qc

In [20]:
qc = teleport()
qc.draw()

┌───┐┌─┐                                            »
q6_0: ────────────■──┤ H ├┤M├────────────────────────────────────────────»
      ┌───┐     ┌─┴─┐└┬─┬┘└╥┘                                            »
q6_1: ┤ H ├──■──┤ X ├─┤M├──╫─────────────────────────────────────────────»
      └───┘┌─┴─┐└───┘ └╥┘  ║   ┌──────   ┌───┐ ───────┐   ┌──────   ┌───┐»
q6_2: ─────┤ X ├───────╫───╫───┤ If-0  ──┤ Z ├  End-0 ├───┤ If-0  ──┤ X ├»
           └───┘       ║   ║   └──╥───   └───┘ ───────┘   └──╥───   └───┘»
                       ║   ║ ┌────╨─────┐               ┌────╨─────┐     »
c6: 3/═════════════════╩═══╩═╡ c6_0=0x1 ╞═══════════════╡ c6_1=0x1 ╞═════»
                       1   0 └──────────┘               └──────────┘     »
«                   
«q6_0: ─────────────
«                   
«q6_1: ─────────────
«       ───────┐ ┌─┐
«q6_2:   End-0 ├─┤M├
«       ───────┘ └╥┘
«c6: 3/═══════════╩═
«                 2

In [17]:
sim = AerSimulator()
compiled = transpile(qc, sim)
result = (sim.run(compiled, shots=10000000)).result()
print(result.get_counts())

{'011': 2500690, '010': 2500753, '000': 2499411, '001': 2499146}


En principio la siguiente versión no debiera funcionar pero quiero tenerla para comprobarlo yo

In [21]:
def teleport():
    c = ClassicalRegister(2)
    qreg = QuantumRegister(3)
    # se crea un registro clásico para poder acceder a los valores individuales de los bits
    qc = QuantumCircuit(qreg,c)
    # se crea un circuito con el registro clasico y 3 qubits, para que esten unidos

    qc = Bell(qc,1,2)
    qc.cx(0,1)
    qc.h(0)
    qc.measure([0,1],[0,1])
    #En función del resultado de c0,c1 hay que aplicar una corrección en q0

    if c[1] == 1:
        qc.x(2)
    
    if c[0] == 1:
        qc.z(2)

    return qc